# Stochastic interpolant demo: unit Gaussian → 2-component mixture

A minimal, self-contained illustration of the `eesi` interpolant framework in 2D.

We learn a generative map from a simple **base** distribution to a more complex
**target**:

- **Base** `p_0`: a single 2D Gaussian, `N(0, I)` (unit variance, zero covariance).
- **Target** `p_1`: a mixture of two Gaussians with non-zero (tilted) covariances.

The `EESI` model trains a drift field `b(t, x)` and a score field `s(t, x)` along
the stochastic interpolant

$$x_t = \alpha(t)\,x_0 + \beta(t)\,x_1 + \gamma(t)\,z,\qquad z\sim N(0, I),$$

then integrates the learned probability-flow ODE from `t=0` (base) to `t=1`
(target). This notebook covers **setup & instantiation**, the **training loop**,
**trajectory sampling**, and **contour plotting**.

In [ ]:
# --- imports & environment ---
import math
import sys
import pathlib

import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.distributions import MultivariateNormal, MixtureSameFamily, Categorical

# Make the `eesi` package importable whether the notebook is launched from the
# repo root or from this experiments/ folder (no install required).
_cwd = pathlib.Path.cwd()
for _root in (_cwd, _cwd.parent):
    if (_root / "eesi" / "__init__.py").exists():
        sys.path.insert(0, str(_root))
        break
from eesi import EESI, TimeMLP

%matplotlib inline
torch.manual_seed(0)
device = torch.device("cpu")

## 1. Base and target distributions

The base is standard normal. The target is a two-component Gaussian mixture whose
components are rotated (correlated) ellipses.

In [ ]:
# --- base: single 2D Gaussian, unit variance, zero covariance ---
base = MultivariateNormal(torch.zeros(2, device=device), torch.eye(2, device=device))

# --- target: mixture of two Gaussians with non-zero covariance ---
def rot_cov(sx, sy, theta):
    """Covariance of an axis-aligned (sx, sy) Gaussian rotated by `theta`."""
    c, s = math.cos(theta), math.sin(theta)
    R = torch.tensor([[c, -s], [s, c]])
    S = torch.diag(torch.tensor([sx ** 2, sy ** 2]))
    return R @ S @ R.T

means = torch.tensor([[-2.5, 2.0], [2.5, -1.5]], device=device)
covs = torch.stack([
    rot_cov(1.4, 0.4, math.pi / 5),
    rot_cov(1.1, 0.5, -math.pi / 6),
]).to(device)
weights = torch.tensor([0.5, 0.5], device=device)

target = MixtureSameFamily(
    Categorical(weights),
    MultivariateNormal(means, covariance_matrix=covs),
)

# Sanity check: draw and scatter samples from each distribution.
x0_vis = base.sample((2000,))
x1_vis = target.sample((2000,))

fig, ax = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
ax[0].scatter(x0_vis[:, 0], x0_vis[:, 1], s=4, alpha=0.3, c="steelblue")
ax[0].set_title("base  $p_0 = N(0, I)$")
ax[1].scatter(x1_vis[:, 0], x1_vis[:, 1], s=4, alpha=0.3, c="crimson")
ax[1].set_title("target  $p_1$ (2-Gaussian mixture)")
for a in ax:
    a.set_aspect("equal"); a.set_xlim(-6, 6); a.set_ylim(-6, 6)
plt.show()

## 2. Model & sampler setup

`EESI` wraps two `TimeMLP` fields — the drift `net_b` and the score `net_s`.

- `path="linear"` uses the straight-line interpolant `x_t = (1-t) x_0 + t x_1`.
- `gamma="quad"` adds the smooth latent-noise schedule `γ(t) = t(1-t)`, so the
  score is trained with the cheap antithetic denoising objective (no divergence
  estimate). Try `gamma="sqrt"` or `gamma="none"`, or `path="trig"`/`"encdec"`.

In [ ]:
net_b = TimeMLP(d=2, hidden=128, n_layers=4, activation="silu")
net_s = TimeMLP(d=2, hidden=128, n_layers=4, activation="silu")

model = EESI(
    net_b, net_s,
    d=2,
    path="linear",   # "linear" | "trig" | "encdec"
    gamma="quad",    # "none" | "quad" | "sqrt"
    gamma_scale=1.0,
).to(device)

opt = torch.optim.Adam(model.parameters(), lr=2e-3)
print(model)

## 3. Training loop

Each step draws a fresh minibatch of `(x_0, x_1)` pairs, forms the interpolant
internally, and returns the drift/score losses. We minimise their sum.

In [ ]:
n_iters = 1500
batch = 512

hist_b, hist_s = [], []
for it in range(n_iters):
    x0 = base.sample((batch,))
    x1 = target.sample((batch,))

    losses = model.loss(x1, x0)          # {"b": drift loss, "s": score loss}
    loss = losses["b"] + losses["s"]

    opt.zero_grad()
    loss.backward()
    opt.step()

    hist_b.append(losses["b"].item())
    hist_s.append(losses["s"].item())
    if it % 200 == 0 or it == n_iters - 1:
        print(f"iter {it:4d}  loss_b={hist_b[-1]:+.3f}  loss_s={hist_s[-1]:+.3f}")

In [ ]:
# Training curves (smoothed).
def smooth(v, k=25):
    v = np.asarray(v)
    if len(v) < k:
        return v
    return np.convolve(v, np.ones(k) / k, mode="valid")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(smooth(hist_b), label="drift loss  (b)")
ax.plot(smooth(hist_s), label="score loss  (s)")
ax.set_xlabel("iteration"); ax.set_ylabel("loss (smoothed)")
ax.legend(); ax.set_title("training losses")
plt.show()

## 4. Trajectory sampling

Integrate the learned probability-flow ODE `dx/dt = b(t, x)` from `t=0` to `t=1`
with Heun's method, recording the full path so we can visualise how base samples
flow to the target.

In [ ]:
@torch.no_grad()
def sample_ode_traj(model, x0, n_steps=100, method="heun"):
    """Integrate dx/dt = net_b(t, x), returning the trajectory [n_steps+1, B, d]."""
    x = x0.clone()
    dt = 1.0 / n_steps
    B = x.shape[0]
    traj = [x.clone()]
    for t in torch.linspace(0.0, 1.0 - dt, n_steps, device=x.device):
        t_b = t.expand(B)
        v1 = model.net_b(t_b, x)
        if method == "euler":
            x = x + dt * v1
        else:  # heun
            v2 = model.net_b((t + dt).expand(B), x + dt * v1)
            x = x + 0.5 * dt * (v1 + v2)
        traj.append(x.clone())
    return torch.stack(traj)

x0 = base.sample((2000,))
traj = sample_ode_traj(model, x0, n_steps=100)   # [101, 2000, 2]
gen = traj[-1]                                    # generated samples at t=1

# How well do generated samples match the target? (mean log-density)
print("mean target log-prob — generated:", float(target.log_prob(gen).mean()))
print("mean target log-prob — true     :", float(target.log_prob(target.sample((2000,))).mean()))
# For comparison, the model also exposes ready-made samplers:
#   x1_hat = model.sample_ode(x0, n_steps=100)          # ODE (drift only)
#   x1_hat = model.sample_sde(x0, n_steps=200, eps=0.1) # SDE (drift + score)

## 5. Contour plotting

Left: the analytic target density with generated samples overlaid.
Right: the target density contours with a handful of ODE trajectories flowing
from the base (black) to the target (crimson).

In [ ]:
# Evaluate the target density on a grid.
lo, hi, n = -6, 6, 200
xs = torch.linspace(lo, hi, n)
ys = torch.linspace(lo, hi, n)
gx, gy = torch.meshgrid(xs, ys, indexing="xy")
grid = torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=-1)
dens = target.log_prob(grid).exp().reshape(n, n)

fig, ax = plt.subplots(1, 2, figsize=(13, 6))

# (a) density + generated samples
ax[0].contourf(gx, gy, dens, levels=25, cmap="Blues")
ax[0].scatter(gen[:, 0], gen[:, 1], s=3, c="crimson", alpha=0.25)
ax[0].set_title("target density + generated samples")

# (b) trajectories base -> target
ax[1].contour(gx, gy, dens, levels=12, cmap="Blues", linewidths=0.8)
idx = torch.randperm(gen.shape[0])[:40]
for j in idx:
    ax[1].plot(traj[:, j, 0], traj[:, j, 1], c="gray", lw=0.6, alpha=0.6)
ax[1].scatter(x0[idx, 0], x0[idx, 1], s=14, c="k", label="$x_0$ (base)", zorder=3)
ax[1].scatter(gen[idx, 0], gen[idx, 1], s=14, c="crimson", label="$\hat{x}_1$ (generated)", zorder=3)
ax[1].legend(loc="upper right")
ax[1].set_title("ODE trajectories: base → target")

for a in ax:
    a.set_aspect("equal"); a.set_xlim(lo, hi); a.set_ylim(lo, hi)
plt.show()

### Next steps

- Swap the interpolant with `path="trig"` or `path="encdec"`, or change the
  latent schedule via `gamma="sqrt"` / `gamma="none"`.
- Sample the reverse-time **SDE** with `model.sample_sde(x0, n_steps=200, eps=0.1)`
  (uses both the drift and the learned score).
- Push the dimension `d` higher and swap in a richer target.